In [0]:
import time

import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = 'car_workshop'
LAB = f'{CATALOG}.lab'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {LAB}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {LAB}.files')
LAB_DIR = f'/Volumes/{CATALOG}/lab/files'


def timed(label, fn):
    t0 = time.time()
    result = fn()
    print(f'{label}: {time.time() - t0:.1f}s')
    return result


print(f'lab schema: {LAB}, lab volume: {LAB_DIR}')

dbutils.widgets.dropdown("Is cluster mode?", "False", ["True","False"])
is_cluster_mode = dbutils.widgets.get("Is cluster mode?")

In [0]:
# how skewed is customer_id? (run this BEFORE any expensive join)
# good way to check what is going on with data
# NOTE: check the distribution of the key you are about to JOIN / GROUP BY on -

# IMPORTANT: that key comes from the QUERY (business question), not from the table's storage

# partitioning. Storage layout is a separate concern -> next cell.

trx = spark.table(f"{CATALOG}.fact.fact_sales_transactions")

dist = trx.groupBy("customer_id").agg(F.count("*").alias("cnt"))
# F.count("*") / F.count(F.lit(1)) — counts all rows, including NULLs.
# F.count("column_name") — counts only non-NULL values in that column within each group.
# there is no count() without argument in PySpark, unlike SQL. You can use F.count(F.lit(1)) to count all rows, including NULLs.

# in this scenario null = walk-in customer, so we want to count them as well.

stats = dist.select(
    F.max("cnt").alias("max_cnt"),
    F.expr("percentile_approx(cnt, 0.5)").alias("median_cnt"),
    # percentile_approx is an approximate version of percentile, which is faster and uses less memory than the exact version. 
    # It is suitable for large datasets where exact percentiles are not required.
    # more in ./learning/pyspark/percentile_approx.md
).first()

display(stats)

print(
    f"hottest key: {stats['max_cnt']:,} rows | median: {stats['median_cnt']:,} "
    f"| skew ratio: {stats['max_cnt'] / stats['median_cnt']:.0f}x"
)
print("rule of thumb: <5x fine, >20x AQE may not be enough -> broadcast / salting")

display(dist.orderBy(F.desc("cnt")).limit(10))  # the NULL row dominates

In [0]:
print('=== NARROW (filter/select) - no Exchange ===')
trx.filter(F.col('payment_method') == 'card').select('transaction_id').explain()

print('=== WIDE (groupBy) - Exchange hashpartitioning ===')
# groupby is wide because keys are spread in different partitions ? 
trx.groupBy('location_id').count().explain()

In [0]:
if is_cluster_mode == 'False':
    print("Skipping this section while it's not for serverless")
else:
    # =====================================================================
    # CLASSIC CLUSTER ONLY (session with mentor) - FAILS on serverless:
    #   - spark.conf.set(...) used below is not on the serverless allowlist
    #   - .rdd API is not supported on serverless
    #   Serverless-friendly variant: NEXT CELL.
    # =====================================================================

    # spark.sql.shuffle.partitions - the most important knob (default 200)
    # target ~128-256 MB per partition after shuffle
    spark.conf.set('spark.sql.adaptive.enabled', 'false')   # show the raw effect first

    spark.conf.set('spark.sql.shuffle.partitions', 8)
    print('shuffle.partitions=8   ->', trx.groupBy('customer_id').count().rdd.getNumPartitions(), 'partitions')

    spark.conf.set('spark.sql.shuffle.partitions', 400)
    print('shuffle.partitions=400 ->', trx.groupBy('customer_id').count().rdd.getNumPartitions(), 'partitions')

    spark.conf.set('spark.sql.adaptive.enabled', 'true')    # AQE coalesces small partitions back
    print('with AQE coalesce      ->', trx.groupBy('customer_id').count().rdd.getNumPartitions(), 'partitions')
    spark.conf.set('spark.sql.shuffle.partitions', 'auto')

In [0]:
# SERVERLESS variant - shuffle partitions are engine-managed (auto AQE), but you
# can still OBSERVE how many partitions a shuffle produced - without the RDD API:
def n_partitions(df):
    """Count distinct physical partitions the rows actually landed in."""
    return (df.withColumn('_pid', F.spark_partition_id())
              .select(F.countDistinct('_pid')).first()[0])


trx_df = spark.table(f'{CATALOG}.fact.fact_sales_transactions')
print('partitions after groupBy shuffle (engine-chosen):',
      n_partitions(trx_df.groupBy('customer_id').count()))
# manually tuning 8 vs 400 vs AQE coalesce -> classic cluster

In [0]:
if is_cluster_mode == 'False':
    print("Skipping this section while it's not for serverless")
else:
    # =====================================================================
    # CLASSIC CLUSTER ONLY (session with mentor) - FAILS on serverless:
    #   - spark.conf.set(...) used below is not on the serverless allowlist
    #   - .rdd API is not supported on serverless
    #   Serverless-friendly variant: NEXT CELL.
    # =====================================================================

    # join strategies: read them from the plan
    locations = spark.table(f'{CATALOG}.dim.dim_locations')

    print('=== dim below autoBroadcastJoinThreshold -> BroadcastHashJoin (no shuffle) ===')
    trx.join(locations, 'location_id').explain()

    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
    print('=== broadcast disabled -> SortMergeJoin (shuffle BOTH sides) ===')
    trx.join(locations, 'location_id').explain()
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')

    # repartition vs coalesce:
    sample = trx.limit(1_000_000)
    print('repartition(50) ->', sample.repartition(50).rdd.getNumPartitions(), '(full shuffle, round-robin)')
    print('coalesce(4)     ->', sample.coalesce(4).rdd.getNumPartitions(), '(narrow merge, no shuffle)')
    # repartition("key") hashes by key again -> re-creates skew for hot keys!

In [0]:
# SERVERLESS variant - you cannot forbid broadcast, but plans are still readable
# and partition counts observable (n_partitions defined in the previous cell):
trx_df = spark.table(f'{CATALOG}.fact.fact_sales_transactions')
locations_df = spark.table(f'{CATALOG}.dim.dim_locations')

print('=== engine picks BroadcastHashJoin on its own - find it in the plan ===')
trx_df.join(locations_df, 'location_id').explain()

sample_sl = trx_df.limit(1_000_000)
print('repartition(50) ->', n_partitions(sample_sl.repartition(50)), '(full shuffle, round-robin)')
print('coalesce(4)     ->', n_partitions(sample_sl.coalesce(4)), '(narrow merge, no shuffle)')

%md
## 4. Auto Loader

Production ingest lives in `autoloader.ipynb` – here a **sandbox** on the lab volume
to trigger the behaviours you get quizzed on: schema inference, `_rescued_data`,
and the overwrite gotcha. Two separate state locations: `schemaLocation`
(inferred/evolving schema) and `checkpointLocation` (which files are done).

In [0]:
import json as pyjson

LANDING = f'{LAB_DIR}/landing/orders'
SCHEMA_LOC = f'{LAB_DIR}/_schemas/orders'
CHK_ORDERS = f'{LAB_DIR}/_checkpoints/orders'

# clean start (lab volume only - safe)
dbutils.fs.rm(f'{LAB_DIR}/landing', True)
dbutils.fs.rm(SCHEMA_LOC, True)
dbutils.fs.rm(CHK_ORDERS, True)
spark.sql(f'DROP TABLE IF EXISTS {LAB}.orders_bronze')

batch1 = [{'order_id': i, 'amount': round(10 + i * 1.5, 2)} for i in range(1, 6)]
dbutils.fs.put(f'{LANDING}/batch1.json', '\n'.join(pyjson.dumps(r) for r in batch1), True)


def ingest_orders():
    (spark.readStream.format('cloudFiles')
        .option('cloudFiles.format', 'json')
        .option('cloudFiles.schemaLocation', SCHEMA_LOC)
        .option('cloudFiles.schemaEvolutionMode', 'rescue')   # never fail: misfits -> _rescued_data
        .option('cloudFiles.schemaHints', 'order_id BIGINT, amount DOUBLE')
        .load(LANDING)
        .writeStream
        .option('checkpointLocation', CHK_ORDERS)
        .trigger(availableNow=True)
        .toTable(f'{LAB}.orders_bronze')
        .awaitTermination())


ingest_orders()
display(spark.table(f'{LAB}.orders_bronze'))

In [0]:
# batch 2: a NEW column and a BROKEN type -> with rescue mode nothing fails,
# misfits land in _rescued_data as JSON. Monitor it: non-null = source changed format!
batch2 = [
    {'order_id': 6, 'amount': 99.9, 'currency': 'PLN'},   # unexpected column
    {'order_id': 'oops-a-string', 'amount': 12.3},        # type mismatch
]
dbutils.fs.put(f'{LANDING}/batch2.json', '\n'.join(pyjson.dumps(r) for r in batch2), True)

ingest_orders()
display(spark.table(f'{LAB}.orders_bronze').orderBy(F.col('order_id').asc_nulls_last()))

In [0]:
# the overwrite gotcha: Auto Loader tracks files BY NAME - overwriting an already
# processed file is silently IGNORED (classic "why is my new data missing")
batch1_fixed = [{'order_id': i, 'amount': 999.99} for i in range(1, 6)]
dbutils.fs.put(f'{LANDING}/batch1.json', '\n'.join(pyjson.dumps(r) for r in batch1_fixed), True)

before = spark.table(f'{LAB}.orders_bronze').count()
ingest_orders()
after = spark.table(f'{LAB}.orders_bronze').count()
print(f'rows before: {before}, after re-ingest of overwritten file: {after} (no change!)')
print('fix: cloudFiles.allowOverwrites=true, or write new files instead of overwriting')
# related options: cloudFiles.backfillInterval (safety net for missed notifications),
# cloudFiles.maxFilesPerTrigger / maxBytesPerTrigger (rate limiting)